In [ ]:
# ==============================================================================
# PROJECT: Telecom Customer Churn Prediction using AI
# DESCRIPTION: Data cleaning, EDA, and Machine Learning to predict customer churn.
# ==============================================================================

# 1. IMPORTING REQUIRED LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2. LOADING THE DATASET
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Dataset Shape:", df.shape)

# 3. DATA CLEANING & PREPROCESSING
# Safely convert 'TotalCharges' to numeric, forcing any strings to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

# Drop 'customerID' as it carries no predictive value
df.drop('customerID', axis=1, inplace=True)

# 4. EXPLORATORY DATA ANALYSIS (EDA) & VISUALIZATION
plt.style.use('seaborn-v0_8-darkgrid')

# Visualization: Churn Count (Fixed Seaborn FutureWarning)
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn', hue='Churn', palette='Set2', legend=False)
plt.title("Distribution of Customer Churn")
plt.show()

# Visualization: Churn by Contract Type
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Contract', hue='Churn', palette='viridis')
plt.title("Customer Churn based on Contract Type")
plt.show()

# 5. FEATURE ENGINEERING
X = df.drop('Churn', axis=1)
y = df['Churn']

# One-Hot Encode categorical features (Fixed boolean cast bug)
categorical_features = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=categorical_features, drop_first=True, dtype=int)

# Encode the target variable (Yes/No to 1/0)
encoder = LabelEncoder()
y = encoder.fit_transform(y)

# Split data into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# (Note: StandardScaler removed. Tree-based models do not require scaling.)

# 6. AI / MACHINE LEARNING MODEL BUILDING
# Fixed imbalanced classes by adding class_weight='balanced'
ai_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
ai_model.fit(X_train, y_train)

# Make predictions on the test set
predictions = ai_model.predict(X_test)

# 7. MODEL EVALUATION & INSIGHTS
accuracy = accuracy_score(y_test, predictions)
print(f"\nAI Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

# Confusion Matrix Visualization
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt='d', cmap='Blues')
plt.title("AI Model Confusion Matrix")
plt.xlabel("Predicted Label (0 = Retained, 1 = Churned)")
plt.ylabel("True Label")
plt.show()